In [0]:
# Spark MLlib — Distributed ML at Scale!
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler,
    StandardScaler,
    StringIndexer
)
from pyspark.ml.classification import (
    RandomForestClassifier,
    LogisticRegression
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import (
    CrossValidator,
    ParamGridBuilder
)
from pyspark.sql import functions as F

In [0]:
# 1. Create dataset
data = [
    (0, 1.2, 3.4, 2.1, "A", 1.0),
    (1, 2.1, 1.2, 3.4, "B", 0.0),
    (2, 3.4, 2.1, 1.2, "A", 1.0),
    (3, 1.1, 3.3, 2.2, "C", 0.0),
    (4, 2.3, 1.4, 3.1, "B", 1.0),
    (5, 3.2, 2.3, 1.4, "A", 0.0),
    (6, 1.4, 3.1, 2.3, "C", 1.0),
    (7, 2.2, 1.3, 3.2, "B", 0.0),
] * 20  # repeat for more data

df = spark.createDataFrame(
    data,
    ["id", "f1", "f2", "f3", "category", "label"]
)

In [0]:
# 2. Feature engineering pipeline
indexer = StringIndexer(
    inputCol="category",
    outputCol="category_idx"
)
assembler = VectorAssembler(
    inputCols=["f1", "f2", "f3", "category_idx"],
    outputCol="features_raw"
)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features"
)
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=10
)

In [0]:
# 3. Build ML Pipeline
pipeline = Pipeline(stages=[
    indexer, assembler, scaler, rf
])

In [0]:
# 4. Train/test split
train, test = df.randomSplit([0.8, 0.2],
                              seed=42)
print(f"Train size: {train.count()}")
print(f"Test size:  {test.count()}")

Train size: 130
Test size:  30


In [0]:
# 5. Fit pipeline
model = pipeline.fit(train)
predictions = model.transform(test)
predictions.select(
    "label", "prediction", "probability"
).show(5)

+-----+----------+-----------+
|label|prediction|probability|
+-----+----------+-----------+
|  1.0|       1.0|  [0.0,1.0]|
|  0.0|       0.0|  [1.0,0.0]|
|  1.0|       1.0|  [0.0,1.0]|
|  1.0|       1.0|  [0.0,1.0]|
|  1.0|       1.0|  [0.0,1.0]|
+-----+----------+-----------+
only showing top 5 rows


In [0]:
# 6. Evaluate
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator.evaluate(predictions)
print(f"\nAccuracy: {accuracy:.4f}")


Accuracy: 1.0000


In [0]:
# 7. CrossValidator — hyperparameter tuning!
import os
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/workspace/default/ml_checkpoints'

rf_tuned = RandomForestClassifier(
    featuresCol="features", labelCol="label"
)
pipeline_tune = Pipeline(stages=[
    indexer, assembler, scaler, rf_tuned
])
paramGrid = (ParamGridBuilder()
    .addGrid(rf_tuned.numTrees, [10, 20])
    .addGrid(rf_tuned.maxDepth, [3, 5])
    .build()
)
cv = CrossValidator(
    estimator=pipeline_tune,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)
cv_model = cv.fit(train)
best_acc = evaluator.evaluate(
    cv_model.transform(test)
)
print(f"Best CV Accuracy: {best_acc:.4f}")
print("Spark MLlib Pipeline complete!")

Best CV Accuracy: 1.0000
Spark MLlib Pipeline complete!
